In [ ]:

# =============================================================================
# GRAPH + NODE DATA FEASIBILITY AUDIT — COMPLETE UPDATED CELL
#
# Data: Original UNSW-NB15 raw flow dataset
# Purpose: Verify feasibility of Node-aware LSTM and Graph + LSTM
#
# READ ONLY
# NO TRAINING
# NO DATASET MODIFICATION
# NO FROZEN V2 TEST ACCESS
# =============================================================================

import io
import numpy as np
import pandas as pd

from google.colab import auth
import google.auth
from google.auth.transport.requests import AuthorizedSession

# =============================================================================
# 1. GOOGLE DRIVE AUTHENTICATION
# =============================================================================

auth.authenticate_user()

creds, _ = google.auth.default()
session = AuthorizedSession(creds)

# Verified Google Drive file IDs
FEATURES_ID = "1AjSW9oYM0Oo59xrWiVgI2_KB8RpS4SMo"
FLOW_ID = "1F38lGA4Rdkl_Ni47EI9QOWuwqmNIteHe"

print("=" * 85)
print("GRAPH + NODE DATA FEASIBILITY AUDIT")
print("=" * 85)


# =============================================================================
# 2. READ FILE CONTENT USING GOOGLE DRIVE API
# =============================================================================

def read_drive_bytes(file_id, max_bytes):

    url = (
        "https://www.googleapis.com/drive/v3/files/"
        f"{file_id}"
    )

    headers = {
        "Range": f"bytes=0-{max_bytes - 1}"
    }

    # Stream data to avoid loading the whole large CSV
    response = session.get(
        url,
        params={"alt": "media"},
        headers=headers,
        timeout=120,
        stream=True
    )

    try:
        response.raise_for_status()

        data = bytearray()

        for chunk in response.iter_content(
            chunk_size=256 * 1024
        ):
            if not chunk:
                continue

            remaining = max_bytes - len(data)

            data.extend(chunk[:remaining])

            if len(data) >= max_bytes:
                break

        result = bytes(data)

    finally:
        response.close()

    if not result:
        raise RuntimeError(
            f"Empty response received for file {file_id}"
        )

    # Guard against receiving an HTML error page
    if result[:100].lstrip().lower().startswith(
        (b"<!doctype html", b"<html")
    ):
        raise RuntimeError(
            "Google Drive returned HTML instead of CSV. "
            "Check the authenticated Google account."
        )

    return result


# =============================================================================
# 3. LOAD OFFICIAL UNSW-NB15 COLUMN DEFINITIONS
# =============================================================================

print("\n1. LOADING UNSW FEATURE DEFINITIONS")

feature_bytes = read_drive_bytes(
    FEATURES_ID,
    max_bytes=100_000
)

# UNSW feature descriptions contain Windows-1252 characters
features = pd.read_csv(
    io.BytesIO(feature_bytes),
    encoding="cp1252",
    skipinitialspace=True
)

features.columns = (
    features.columns
    .astype(str)
    .str.strip()
)

if "Name" not in features.columns:
    raise ValueError(
        f"Expected 'Name' column. Found: "
        f"{features.columns.tolist()}"
    )

names = (
    features["Name"]
    .astype(str)
    .str.strip()
    .tolist()
)

print("Total feature definitions:", len(names))
print("First 10 columns:", names[:10])

if len(names) != 49:
    raise ValueError(
        f"Expected 49 UNSW columns, found {len(names)}"
    )

print("Feature schema: PASS")


# =============================================================================
# 4. READ RAW NETWORK FLOWS
# =============================================================================

print("\n2. LOADING RAW UNSW NETWORK FLOWS")

# Read only a partial sample, not the entire large dataset
flow_bytes = read_drive_bytes(
    FLOW_ID,
    max_bytes=12_000_000
)

# Remove any incomplete final CSV record
last_newline = flow_bytes.rfind(b"\n")

if last_newline == -1:
    raise RuntimeError(
        "No complete CSV records found in downloaded sample."
    )

flow_bytes = flow_bytes[:last_newline + 1]

df = pd.read_csv(
    io.BytesIO(flow_bytes),
    header=None,
    names=names,
    nrows=10000,
    encoding="cp1252",
    low_memory=False,
    skipinitialspace=True
)

df.columns = df.columns.str.strip()

print("Rows examined:", len(df))
print("Columns:", len(df.columns))
print("Raw flow loading: PASS")


# =============================================================================
# 5. CHECK NETWORK GRAPH FIELDS
# =============================================================================

print("\n3. REQUIRED GRAPH FIELDS")
print("-" * 65)

required = [
    "srcip",
    "dstip",
    "sport",
    "dsport",
    "proto",
    "Stime",
    "Ltime",
    "Label",
    "attack_cat"
]

missing = []

for col in required:

    found = col in df.columns

    print(
        f"{col:15s} : "
        f"{'FOUND' if found else 'MISSING'}"
    )

    if not found:
        missing.append(col)

print("\nRequired fields available:",
      len(required) - len(missing),
      "/",
      len(required))


# =============================================================================
# 6. ANALYZE NETWORK NODES AND CONNECTIONS
# =============================================================================

print("\n4. GRAPH STRUCTURE")
print("-" * 65)

if {"srcip", "dstip"}.issubset(df.columns):

    src = (
        df["srcip"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    dst = (
        df["dstip"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    nodes = pd.concat(
        [src, dst],
        ignore_index=True
    ).dropna().unique()

    edges = pd.DataFrame({
        "srcip": src,
        "dstip": dst
    }).dropna().drop_duplicates()

    print("Unique network nodes:", len(nodes))
    print("Unique directed connections:", len(edges))

    print("\nExample node IDs:")
    print(list(nodes[:5]))

    print("\nExample network connections:")
    print(edges.head(5).to_string(index=False))

else:
    print("Node fields unavailable.")


# =============================================================================
# 7. ANALYZE TEMPORAL INFORMATION
# =============================================================================

print("\n5. TIMESTAMP ANALYSIS")
print("-" * 65)

if "Stime" in df.columns:

    times = pd.to_numeric(
        df["Stime"],
        errors="coerce"
    )

    valid_times = times.dropna()

    print("Valid timestamps:", len(valid_times))
    print("Unique timestamps:", valid_times.nunique())

    if len(valid_times) > 0:

        print("Earliest timestamp:", valid_times.min())
        print("Latest timestamp:", valid_times.max())

        print(
            "Time span in minutes:",
            round(
                (valid_times.max() - valid_times.min()) / 60,
                3
            )
        )

        print(
            "Earliest UTC time:",
            pd.to_datetime(
                valid_times.min(),
                unit="s",
                utc=True
            )
        )

        print(
            "Latest UTC time:",
            pd.to_datetime(
                valid_times.max(),
                unit="s",
                utc=True
            )
        )

else:
    print("Start timestamps unavailable.")


# =============================================================================
# 8. ANALYZE ATTACK LABELS
# =============================================================================

print("\n6. ATTACK LABEL DISTRIBUTION")
print("-" * 65)

if "Label" in df.columns:

    print(
        df["Label"]
        .value_counts(dropna=False)
        .to_string()
    )

else:
    print("Binary attack labels unavailable.")


print("\n7. ATTACK FAMILY DISTRIBUTION")
print("-" * 65)

if "attack_cat" in df.columns:

    print(
        df["attack_cat"]
        .astype("string")
        .str.strip()
        .value_counts(dropna=False)
        .to_string()
    )

else:
    print("Attack family labels unavailable.")


# =============================================================================
# 9. CHECK WHETHER TEMPORAL GRAPHS ARE FEASIBLE
# =============================================================================

print("\n8. TEMPORAL GRAPH FEASIBILITY")
print("-" * 65)

has_nodes = {"srcip", "dstip"}.issubset(df.columns)

has_time = (
    "Stime" in df.columns
    and pd.to_numeric(
        df["Stime"], errors="coerce"
    ).notna().any()
)

has_labels = "Label" in df.columns

if has_nodes:
    print("Node identities: AVAILABLE")
else:
    print("Node identities: NOT AVAILABLE")

if has_nodes:
    print("Graph edges: AVAILABLE")
else:
    print("Graph edges: NOT AVAILABLE")

if has_time:
    print("Temporal ordering: AVAILABLE IN SAMPLE")
else:
    print("Temporal ordering: NOT AVAILABLE")

if has_labels:
    print("Attack labels: AVAILABLE")
else:
    print("Attack labels: NOT AVAILABLE")

if has_nodes and has_time and has_labels:

    print("\nSCHEMA FEASIBILITY: PASS")

    print(
        "Raw fields support further investigation "
        "of temporal graph construction."
    )

    print(
        "Future-onset forecasting labels and "
        "chronological validity are NOT YET VERIFIED."
    )

else:

    print("\nSCHEMA FEASIBILITY: INCOMPLETE")

    print(
        "Some required raw fields are missing."
    )


# =============================================================================
# 10. FINAL REPORT
# =============================================================================

print("\n" + "=" * 85)
print("GRAPH + NODE DATA AUDIT COMPLETE")
print("=" * 85)

print("Raw dataset: UNSW-NB15")
print("Rows analyzed:", len(df))
print("Original dataset modified: NO")
print("Model trained: NO")
print("Frozen V2 TEST accessed: NO")
print("Next stage: Temporal graph and label feasibility checks")

print("=" * 85)


In [ ]:

# ================================================================
# GRAPH DATA — CELL 2
# FULL-FILE TIMELINE AND FUTURE-ATTACK FEASIBILITY AUDIT
# READ ONLY — NO MODEL TRAINING — NO FROZEN V2 TEST ACCESS
# ================================================================

from pathlib import Path
from collections import defaultdict
import io
import numpy as np
import pandas as pd

from google.colab import auth
import google.auth
from google.auth.transport.requests import AuthorizedSession

auth.authenticate_user()
creds, _ = google.auth.default()
session = AuthorizedSession(creds)

FEATURES_ID = "1AjSW9oYM0Oo59xrWiVgI2_KB8RpS4SMo"
FLOW_ID = "1F38lGA4Rdkl_Ni47EI9QOWuwqmNIteHe"

print("=" * 75)
print("GRAPH DATA — FULL CHRONOLOGICAL FEASIBILITY AUDIT")
print("=" * 75)

# 1. Read official column names
url = f"https://www.googleapis.com/drive/v3/files/{FEATURES_ID}"
r = session.get(url, params={"alt": "media"}, timeout=120)
r.raise_for_status()

features = pd.read_csv(
    io.BytesIO(r.content),
    encoding="cp1252"
)
features.columns = features.columns.str.strip()
names = features["Name"].astype(str).str.strip().tolist()
assert len(names) == 49

# 2. Locate local file or download to temporary Colab storage
mounted_file = Path(
    "/content/drive/MyDrive/network-attack-forecasting/"
    "Dataset_Audit_v2/raw/UNSW_NB15/UNSW-NB15_1.csv"
)

local_file = Path("/content/UNSW_NB15_1_GRAPH_AUDIT.csv")

if mounted_file.is_file():
    flow_path = mounted_file
    print("Using mounted original file:", flow_path)
else:
    flow_path = local_file

    expected_size = 168979718

    if not (
        local_file.is_file()
        and local_file.stat().st_size == expected_size
    ):
        print("Downloading raw flow file to temporary Colab storage...")

        flow_url = (
            f"https://www.googleapis.com/drive/v3/files/{FLOW_ID}"
        )

        tmp_path = local_file.with_suffix(".part")

        with session.get(
            flow_url,
            params={"alt": "media"},
            stream=True,
            timeout=180
        ) as response:

            response.raise_for_status()
            downloaded = 0
            last_report = 0

            with open(tmp_path, "wb") as output:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if not chunk:
                        continue

                    output.write(chunk)
                    downloaded += len(chunk)

                    if downloaded - last_report >= 25_000_000:
                        print(
                            "Downloaded:",
                            round(downloaded / 1e6, 1),
                            "MB"
                        )
                        last_report = downloaded

        if tmp_path.stat().st_size != expected_size:
            raise RuntimeError(
                "Incomplete download: expected "
                f"{expected_size} bytes, received "
                f"{tmp_path.stat().st_size}. "
                "The temporary file was not accepted."
            )

        tmp_path.replace(local_file)

        print("Download complete.")

print("\nReading full raw file in chunks...")

# 3. Stream useful fields without loading the full CSV into RAM
minute_stats = defaultdict(lambda: [0, 0])

total_rows = 0
valid_rows = 0
invalid_rows = 0
out_of_order_pairs = 0
previous_time = None
chunk_count = 0

columns_needed = [
    "srcip", "dstip", "Stime",
    "Ltime", "Label", "attack_cat"
]

reader = pd.read_csv(
    flow_path,
    header=None,
    names=names,
    usecols=columns_needed,
    chunksize=200_000,
    encoding="cp1252",
    low_memory=False
)

for chunk in reader:
    chunk_count += 1
    total_rows += len(chunk)

    # Clean BOM corruption in source/destination identifiers
    for col in ["srcip", "dstip"]:
        chunk[col] = (
            chunk[col]
            .astype("string")
            .str.strip()
            .str.replace(
                r"^(?:\ufeff|ï»¿)+",
                "",
                regex=True
            )
        )

    start = pd.to_numeric(
        chunk["Stime"], errors="coerce"
    )

    label = pd.to_numeric(
        chunk["Label"], errors="coerce"
    )

    valid = (
        start.notna()
        & (start > 0)
        & label.isin([0, 1])
    )

    invalid_rows += int((~valid).sum())
    valid_rows += int(valid.sum())

    # Check whether physical CSV row order is chronological
    valid_times = start[start.notna()].to_numpy()

    if len(valid_times):
        out_of_order_pairs += int(
            (np.diff(valid_times) < 0).sum()
        )

        if previous_time is not None:
            out_of_order_pairs += int(
                valid_times[0] < previous_time
            )

        previous_time = valid_times[-1]

    if not valid.any():
        continue

    sample = pd.DataFrame({
        "minute": (
            start.loc[valid] // 60
        ).astype("int64"),
        "attack": (
            label.loc[valid] == 1
        ).astype("int8")
    })

    grouped = sample.groupby("minute")["attack"].agg(
        ["count", "sum"]
    )

    for minute, row in grouped.iterrows():
        minute_stats[int(minute)][0] += int(row["count"])
        minute_stats[int(minute)][1] += int(row["sum"])

    print(
        f"Chunk {chunk_count}: "
        f"{total_rows:,} rows scanned",
        flush=True
    )

# 4. Reconstruct the minute-level timeline
minutes = sorted(minute_stats)
present = set(minutes)

attack_minute = {
    m: minute_stats[m][1] > 0
    for m in minutes
}

print("\n" + "=" * 75)
print("FULL-FILE TIMELINE RESULTS")
print("=" * 75)

print("Rows scanned:", total_rows)
print("Valid labeled rows:", valid_rows)
print("Invalid timestamp/label rows:", invalid_rows)
print("Observed minutes:", len(minutes))
print("Out-of-order adjacent timestamp pairs:", out_of_order_pairs)

if not minutes:
    raise RuntimeError("No usable chronological data found.")

total_span = minutes[-1] - minutes[0] + 1
missing_minutes = total_span - len(minutes)

print("First UTC minute:",
      pd.to_datetime(minutes[0] * 60, unit="s", utc=True))
print("Last UTC minute:",
      pd.to_datetime(minutes[-1] * 60, unit="s", utc=True))

print("Total timeline span (minutes):", total_span)
print("Missing minutes inside span:", missing_minutes)
print(
    "Minutes containing attack-labeled flows:",
    sum(attack_minute.values())
)

# 5. Identify observed attack-minute transitions
# This is only a provisional onset indicator.
onsets = {
    m for m in minutes
    if attack_minute[m]
    and (m - 1) in present
    and not attack_minute[m - 1]
}

print("\nProvisional minute-level attack onsets:", len(onsets))

# 6. Require 10 complete history minutes + 15 future minutes
eligible = []

for m in minutes:
    if not all(
        (m + offset) in present
        for offset in range(-9, 16)
    ):
        continue

    # Avoid forecasting from a minute where an attack
    # is already observed in the aggregate timeline.
    if attack_minute[m]:
        continue

    eligible.append(m)

print("\nFORECASTING WINDOW FEASIBILITY")
print("10-minute history + 15-minute future windows:",
      len(eligible))

for horizon in [5, 10, 15]:
    positive = sum(
        any((m + dt) in onsets for dt in range(1, horizon + 1))
        for m in eligible
    )

    prevalence = (
        positive / len(eligible)
        if eligible else 0.0
    )

    print(
        f"H{horizon}: "
        f"positive onset windows={positive} | "
        f"prevalence={prevalence:.4%}"
    )

print("\n" + "=" * 75)
print("AUDIT COMPLETE")
print("=" * 75)
print("Original UNSW file modified: NO")
print("Frozen V2 dataset modified: NO")
print("Official frozen V2 TEST accessed: NO")
print("Model training performed: NO")
print(
    "NOTE: Minute-level attack transitions are provisional."
)
print(
    "Causal validity and true attack onset labels "
    "still require further verification."
)


In [ ]:

# ================================================================
# GRAPH DATA — CELL 3
# AUDIT ALL FOUR ORIGINAL UNSW-NB15 FLOW FILES
# READ ONLY / NO FROZEN V2 TEST
# ================================================================

from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

from google.colab import auth
import google.auth
from google.auth.transport.requests import AuthorizedSession

auth.authenticate_user()
creds, _ = google.auth.default()
session = AuthorizedSession(creds)

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/"
    "Dataset_Audit_v2/raw/UNSW_NB15"
)

FILES = [
    ("UNSW-NB15_1.csv", "1F38lGA4Rdkl_Ni47EI9QOWuwqmNIteHe"),
    ("UNSW-NB15_2.csv", "1FXYLWnb01fcn-bYGELeZrpoA0Y9lqYK7"),
    ("UNSW-NB15_3.csv", "1jx9op-bKPSQFchT5YnQEZL4S-uxenyOL"),
    ("UNSW-NB15_4.csv", "16OMur9HR0DrIXUve7m1jeMXLVKicfiaV"),
]

# 0-based positions in the verified 49-column UNSW schema
TIME_COLUMN = 28  # Stime
LABEL_COLUMN = 48 # Label

combined = defaultdict(lambda: [0, 0])
file_reports = []

print("=" * 76)
print("UNSW-NB15 — COMPLETE GRAPH FORECASTING TIMELINE AUDIT")
print("=" * 76)

for filename, file_id in FILES:

    print(f"\nScanning: {filename}", flush=True)

    per_file = defaultdict(lambda: [0, 0])
    rows = 0
    invalid = 0

    mounted = ROOT / filename

    if mounted.is_file():
        source = mounted
        response = None
        print("Source: mounted Google Drive")
    else:
        url = f"https://www.googleapis.com/drive/v3/files/{file_id}"
        response = session.get(
            url,
            params={"alt": "media"},
            stream=True,
            timeout=180
        )
        response.raise_for_status()
        response.raw.decode_content = True
        source = response.raw
        print("Source: authenticated Drive stream")

    try:
        reader = pd.read_csv(
            source,
            header=None,
            usecols=[TIME_COLUMN, LABEL_COLUMN],
            encoding="cp1252",
            chunksize=150_000,
            on_bad_lines="error"
        )

        for chunk in reader:

            rows += len(chunk)

            t = pd.to_numeric(
                chunk[TIME_COLUMN], errors="coerce"
            )

            y = pd.to_numeric(
                chunk[LABEL_COLUMN], errors="coerce"
            )

            valid = (
                t.notna()
                & (t > 0)
                & y.isin([0, 1])
            )

            invalid += int((~valid).sum())

            if not valid.any():
                continue

            part = pd.DataFrame({
                "minute": (
                    t.loc[valid] // 60
                ).astype("int64"),
                "attack": (
                    y.loc[valid] == 1
                ).astype("int8")
            })

            grouped = part.groupby("minute")["attack"].agg(
                ["count", "sum"]
            )

            for minute, record in grouped.iterrows():
                m = int(minute)
                per_file[m][0] += int(record["count"])
                per_file[m][1] += int(record["sum"])
                combined[m][0] += int(record["count"])
                combined[m][1] += int(record["sum"])

    finally:
        if response is not None:
            response.close()

    if not per_file:
        raise RuntimeError(f"No usable records found: {filename}")

    times = sorted(per_file)
    attack_minutes = sum(
        per_file[m][1] > 0 for m in times
    )

    file_reports.append({
        "File": filename,
        "Rows": rows,
        "Invalid": invalid,
        "Minutes": len(times),
        "AttackMinutes": attack_minutes,
        "FirstMinute": times[0],
        "LastMinute": times[-1]
    })

    print(
        f"Rows={rows:,}, minutes={len(times)}, "
        f"attack_minutes={attack_minutes}, invalid={invalid}",
        flush=True
    )

# ================================================================
# COMBINED CHRONOLOGICAL ANALYSIS
# ================================================================

minutes = sorted(combined)
present = set(minutes)

has_attack = {
    m: combined[m][1] > 0
    for m in minutes
}

onsets = {
    m for m in minutes
    if has_attack[m]
    and (m - 1) in present
    and not has_attack[m - 1]
}

# Require complete observed history and future.
# Also require all 10 observed minutes to be attack-free.
eligible = []

for m in minutes:

    if not all(
        (m + offset) in present
        for offset in range(-9, 16)
    ):
        continue

    if any(
        has_attack[m - offset]
        for offset in range(10)
    ):
        continue

    eligible.append(m)

print("\n" + "=" * 76)
print("PER-FILE SUMMARY")
print("=" * 76)

print(pd.DataFrame(file_reports).to_string(index=False))

print("\n" + "=" * 76)
print("COMBINED CHRONOLOGICAL RESULTS")
print("=" * 76)

print("Observed minutes:", len(minutes))
print("Minutes containing attacks:", sum(has_attack.values()))
print("Provisional attack onsets:", len(onsets))
print("Eligible clean-history forecasting windows:", len(eligible))

print(
    "First observed UTC:",
    pd.to_datetime(minutes[0] * 60, unit="s", utc=True)
)
print(
    "Last observed UTC:",
    pd.to_datetime(minutes[-1] * 60, unit="s", utc=True)
)

positive_counts = {}

for horizon in [5, 10, 15]:

    positives = sum(
        any(
            (m + delta) in onsets
            for delta in range(1, horizon + 1)
        )
        for m in eligible
    )

    positive_counts[horizon] = positives

    prevalence = (
        positives / len(eligible)
        if eligible else 0.0
    )

    print(
        f"H{horizon}: "
        f"positive_windows={positives} | "
        f"prevalence={prevalence:.4%}"
    )

print("\n" + "=" * 76)
print("INITIAL FEASIBILITY DECISION")
print("=" * 76)

if (
    len(eligible) >= 100
    and all(positive_counts[h] >= 20 for h in [5, 10, 15])
):
    print("CANDIDATE: Enough positive windows for further audit.")
    print("Still requires independent onset and leakage checks.")
else:
    print("INSUFFICIENT: Do not train a forecasting GNN yet.")
    print("Consider another valid timeline or task definition.")

print("\nNo files modified.")
print("No models trained.")
print("Frozen V2 TEST not accessed.")


In [ ]:

# ================================================================
# GRAPH DATA — CELL 4
# NODE-LEVEL FUTURE ATTACK FEASIBILITY
# Original UNSW-NB15 raw data, read only
# ================================================================

from collections import defaultdict
from pathlib import Path
import pandas as pd
import numpy as np
from google.colab import auth
import google.auth
from google.auth.transport.requests import AuthorizedSession

auth.authenticate_user()
creds, _ = google.auth.default()
session = AuthorizedSession(creds)

FILES = [
    ("UNSW-NB15_1.csv", "1F38lGA4Rdkl_Ni47EI9QOWuwqmNIteHe"),
    ("UNSW-NB15_2.csv", "1FXYLWnb01fcn-bYGELeZrpoA0Y9lqYK7"),
    ("UNSW-NB15_3.csv", "1jx9op-bKPSQFchT5YnQEZL4S-uxenyOL"),
    ("UNSW-NB15_4.csv", "16OMur9HR0DrIXUve7m1jeMXLVKicfiaV"),
]

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/"
    "Dataset_Audit_v2/raw/UNSW_NB15"
)

# Host -> minutes containing traffic
host_active = defaultdict(set)

# Host -> minutes containing normal traffic
host_benign = defaultdict(set)

# Host -> minutes containing attack-labeled traffic
host_attack = defaultdict(set)

global_minutes = set()
total_rows = 0

print("=" * 75)
print("NODE-LEVEL ATTACK ONSET FEASIBILITY AUDIT")
print("=" * 75)

for filename, file_id in FILES:
    mounted = ROOT / filename
    response = None

    if mounted.is_file():
        source = mounted
        print("\nReading mounted file:", filename)
    else:
        url = f"https://www.googleapis.com/drive/v3/files/{file_id}"
        response = session.get(
            url,
            params={"alt": "media"},
            stream=True,
            timeout=180
        )
        response.raise_for_status()
        response.raw.decode_content = True
        source = response.raw
        print("\nStreaming:", filename)

    try:
        reader = pd.read_csv(
            source,
            header=None,
            usecols=[2, 28, 48],
            encoding="cp1252",
            chunksize=150000,
            low_memory=False
        )

        for chunk in reader:
            # 2 = destination IP, 28 = start time, 48 = label
            host = (
                chunk[2].astype("string")
                .str.strip()
                .str.replace(
                    r"^(?:\ufeff|ï»¿)+", "", regex=True
                )
            )

            t = pd.to_numeric(chunk[28], errors="coerce")
            y = pd.to_numeric(chunk[48], errors="coerce")

            valid = (
                host.notna()
                & host.ne("")
                & t.notna()
                & (t > 0)
                & y.isin([0, 1])
            )

            part = pd.DataFrame({
                "host": host.loc[valid].to_numpy(),
                "minute": (t.loc[valid] // 60)
                    .astype("int64").to_numpy(),
                "attack": y.loc[valid]
                    .astype("int8").to_numpy()
            })

            total_rows += len(chunk)

            if part.empty:
                continue

            grouped = part.groupby(
                ["host", "minute"], sort=False
            )["attack"].agg(["min", "max"])

            for (h, minute), values in grouped.iterrows():
                minute = int(minute)
                h = str(h)

                global_minutes.add(minute)
                host_active[h].add(minute)

                if int(values["min"]) == 0:
                    host_benign[h].add(minute)

                if int(values["max"]) == 1:
                    host_attack[h].add(minute)

    finally:
        if response is not None:
            response.close()

    print("Total rows scanned:", f"{total_rows:,}")

# ================================================================
# FIND PROVISIONAL NODE-LEVEL ATTACK ONSETS
# ================================================================

print("\nBuilding host timelines...")

onsets_by_host = defaultdict(set)

for host, attack_minutes in host_attack.items():
    benign_minutes = host_benign[host]

    for m in attack_minutes:
        # Require complete network observation before onset
        if not all(
            (m - k) in global_minutes for k in range(1, 11)
        ):
            continue

        # Require a clean host-specific 10-minute history
        if any(
            (m - k) in attack_minutes for k in range(1, 11)
        ):
            continue

        # Host must have observed benign activity beforehand
        if not any(
            (m - k) in benign_minutes for k in range(1, 11)
        ):
            continue

        onsets_by_host[host].add(m)

# ================================================================
# ELIGIBLE FORECAST DECISION POINTS
# ================================================================

eligible_global = [
    m for m in sorted(global_minutes)
    if all(
        (m + offset) in global_minutes
        for offset in range(-9, 16)
    )
]

counts = {5: 0, 10: 0, 15: 0}
positive_events = {5: set(), 10: set(), 15: set()}
total_examples = 0
hosts_with_examples = set()

for host, active_minutes in host_active.items():
    attack_minutes = host_attack[host]
    benign_minutes = host_benign[host]
    host_onsets = onsets_by_host[host]

    for m in eligible_global:
        # No attack observed during 10-minute input history
        if any(
            (m - k) in attack_minutes for k in range(10)
        ):
            continue

        # Need some actual host activity in input history
        if not any(
            (m - k) in benign_minutes for k in range(10)
        ):
            continue

        total_examples += 1
        hosts_with_examples.add(host)

        for horizon in [5, 10, 15]:
            future_onsets = [
                onset for onset in host_onsets
                if m < onset <= m + horizon
            ]

            if future_onsets:
                counts[horizon] += 1

                for onset in future_onsets:
                    positive_events[horizon].add(
                        (host, onset)
                    )

# ================================================================
# REPORT
# ================================================================

print("\n" + "=" * 75)
print("NODE-LEVEL FEASIBILITY RESULTS")
print("=" * 75)

print("Total rows scanned:", f"{total_rows:,}")
print("Destination nodes:", len(host_active))
print("Nodes with attack-labeled traffic:",
      len(host_attack))

all_onsets = {
    (host, minute)
    for host, mins in onsets_by_host.items()
    for minute in mins
}

print("Provisional node-level onsets:", len(all_onsets))
print("Hosts with usable forecasting history:",
      len(hosts_with_examples))
print("Eligible host-minute forecasting examples:",
      total_examples)

for h in [5, 10, 15]:
    prevalence = (
        counts[h] / total_examples
        if total_examples else 0
    )

    print(
        f"H{h} | "
        f"Positive windows={counts[h]} | "
        f"Distinct reachable onset events="
        f"{len(positive_events[h])} | "
        f"Prevalence={prevalence:.4%}"
    )

print("\n" + "=" * 75)
print("PRELIMINARY DECISION")
print("=" * 75)

if (
    total_examples > 0
    and len(positive_events[15]) >= 20
):
    print("CANDIDATE: Node-level forecasting may be feasible.")
    print("Next: independent events, leakage and split audit.")
else:
    print("INSUFFICIENT for node-level forecasting under this definition.")
    print("Next: choose a better graph-forecasting data source.")

print("\nREAD ONLY: No files modified.")
print("Frozen V2 TEST accessed: NO")
print("No model training performed.")
print("Onsets are provisional, not verified attack campaigns.")
